In [3]:
import torch, os, sys
import numpy as np
torch.backends.cuda.enable_mem_efficient_sdp(False)
torch.backends.cuda.enable_flash_sdp(False)
torch.backends.cuda.enable_math_sdp(True)
import inspect
os.environ['CUDA_VISIBLE_DEVICES'] = '3'
import pandas as pd
from fast_transformers.builders import TransformerEncoderBuilder
from torch.nn import TransformerEncoder, TransformerEncoderLayer
from fast_transformers.masking import FullMask, LengthMask

root_dir = os.path.dirname(os.getcwd())
sys.path.append(root_dir)
import pdb
import torch.nn.functional as F
import random

In [9]:
n_layers = 2
num_heads = 1
embed_dim = 32
n_hid = 128
dropout = 0.1

In [5]:
torch.manual_seed(0)
# Create the builder for our transformers
builder = TransformerEncoderBuilder.from_kwargs(
    n_layers=n_layers,
    n_heads=num_heads,
    query_dimensions=embed_dim // num_heads,
    value_dimensions=embed_dim // num_heads,
    feed_forward_dimensions=n_hid,
    dropout=dropout
)

# Build a transformer with softmax attention
builder.attention_type = "full"
softmax_model = builder.get().to('cuda')

# Build a transformer with linear attention
builder.attention_type = "linear"
linear_model = builder.get().to('cuda')


In [4]:
def generate_associative_recall_batch(seq_len=10, batch_size=32, vocab_size=10, device='cuda'):
    """Generate a batch of associative recall training data with discrete tokens"""
    # Generate random key-value pairs from a limited vocabulary
    # Generate random token IDs
    keys = torch.randint(0, vocab_size // 2, (batch_size, seq_len), device=device)
    values = torch.randint(vocab_size // 2, vocab_size, (batch_size, seq_len), device=device)
    
    token_embeddings = torch.randn(vocab_size, embed_dim, device=device)  # Random embedding matrix
    
    # Project tokens to embedding space
    keys_embedded = token_embeddings[keys]  # [batch, seq_len, embed_dim] 
    values_embedded = token_embeddings[values]  # [batch, seq_len, embed_dim]
    
    # Randomly select query indices
    query_indices = torch.randint(0, seq_len, (batch_size,), device=device)
    queries = keys_embedded[torch.arange(batch_size), query_indices]  # [batch, embed_dim]
    
    # True targets are the values corresponding to the queries
    targets = values[torch.arange(batch_size), query_indices]  # [batch]
    
    # Prepare input sequence: concatenate keys and values, then append query
    input_seq = torch.stack([keys_embedded, values_embedded], dim=2)  # [batch, seq_len, 2, embed_dim]
    input_seq = input_seq.view(batch_size, seq_len*2, embed_dim)  # [batch, seq_len*2, embed_dim]
    input_seq = torch.cat([input_seq, queries.unsqueeze(1)], dim=1)  # [batch, seq_len*2+1, embed_dim]

    return input_seq, targets


In [5]:

def train_model(model, seq_len, vocab_size=10, n_epochs=1000, batch_size=1024, device='cuda'):
    """Train model on associative recall classification task"""
    optimizer = torch.optim.Adam(model.parameters(), lr=5e-4, weight_decay=0.1)
    criterion = torch.nn.CrossEntropyLoss()
    losses = []
    
    for epoch in range(n_epochs):
        model.train()
        input_seq, targets = generate_associative_recall_batch(seq_len, batch_size, vocab_size, device)
        
        # Create attention mask
        attention_mask = FullMask(seq_len*2 + 1)
        
        # Forward pass
        optimizer.zero_grad()

        output = model(input_seq, attn_mask=attention_mask)  # [batch, seq_len*2+1, vocab_size]
        predictions = output[:, -1]  # Take last token predictions [batch, vocab_size]
        
        # Compute loss
        loss = criterion(predictions, targets)
        # Compute accuracy
        acc = (predictions.argmax(dim=-1) == targets).float().mean()
        
        # Backward pass
        loss.backward()
        optimizer.step()
        
        losses.append(loss.item())
        
        if (epoch + 1) % 20 == 0:
            print(f"Epoch {epoch+1}/{n_epochs}, Loss: {loss.item():.6f}, Acc: {acc.item():.6f}")
    
    return np.mean(losses[-10:])  # Return average of last 10 losses

In [6]:
def evaluate_model(model, seq_len, vocab_size=10, n_batches=10, batch_size=32, device='cuda'):
    """Evaluate model on associative recall classification task"""
    model.eval()
    total_acc = 0
    
    with torch.no_grad():
        for _ in range(n_batches):
            input_seq, targets = generate_associative_recall_batch(seq_len, batch_size, vocab_size, device)
            
            attention_mask = FullMask(seq_len*2 + 1)
            
            output = model(input_seq, attn_mask=attention_mask)
            predictions = output[:, -1].argmax(dim=-1)  # [batch]
            
            acc = (predictions == targets).float().mean()
            total_acc += acc.item()
    
    return total_acc / n_batches

In [7]:
# Run experiments
seq_lengths = [20]
vocab_size = 10
results = {'linear': [], 'softmax': []}

print("\nTraining and evaluating models...")
print("\nSequence Length | Linear Acc | Softmax Acc")
print("-" * 45)

for seq_len in seq_lengths:
    # Reset models
    builder.attention_type = "full"
    softmax_model = builder.get().to('cuda')
    builder.attention_type = "linear"
    linear_model = builder.get().to('cuda')
    
    # Train models
    print(f"\nTraining on sequence length {seq_len}:")
    print("Linear attention model:")
    linear_train_loss = train_model(linear_model, seq_len, vocab_size)
    print("\nSoftmax attention model:")
    softmax_train_loss = train_model(softmax_model, seq_len, vocab_size)
    
    # Evaluate models
    linear_eval_acc = evaluate_model(linear_model, seq_len, vocab_size)
    softmax_eval_acc = evaluate_model(softmax_model, seq_len, vocab_size)
    
    results['linear'].append(linear_eval_acc)
    results['softmax'].append(softmax_eval_acc)
    
    print(f"\n{seq_len:14d} | {linear_eval_acc:.6f} | {softmax_eval_acc:.6f}")

# Create DataFrame with results
results_df = pd.DataFrame({
    'seq_length': seq_lengths,
    'linear_acc': results['linear'],
    'softmax_acc': results['softmax']
})

# Plot results
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))
plt.plot(seq_lengths, results['linear'], 'b-o', label='Linear Attention')
plt.plot(seq_lengths, results['softmax'], 'r-o', label='Softmax Attention')
plt.xlabel('Sequence Length')
plt.ylabel('Accuracy')
plt.title('Associative Recall Performance: Linear vs Softmax Attention')
plt.legend()
plt.grid(True)
plt.show()



Training and evaluating models...

Sequence Length | Linear Acc | Softmax Acc
---------------------------------------------

Training on sequence length 20:
Linear attention model:
Epoch 20/1000, Loss: 3.459129, Acc: 0.075195
Epoch 40/1000, Loss: 2.642152, Acc: 0.213867
Epoch 60/1000, Loss: 2.458141, Acc: 0.204102
Epoch 80/1000, Loss: 2.329844, Acc: 0.180664
Epoch 100/1000, Loss: 2.209234, Acc: 0.199219
Epoch 120/1000, Loss: 2.133780, Acc: 0.204102
Epoch 140/1000, Loss: 2.121519, Acc: 0.189453
Epoch 160/1000, Loss: 2.096309, Acc: 0.220703
Epoch 180/1000, Loss: 2.211592, Acc: 0.190430
Epoch 200/1000, Loss: 2.075587, Acc: 0.204102
Epoch 220/1000, Loss: 2.215915, Acc: 0.206055
Epoch 240/1000, Loss: 2.079107, Acc: 0.205078
Epoch 260/1000, Loss: 2.111540, Acc: 0.198242
Epoch 280/1000, Loss: 2.079043, Acc: 0.202148
Epoch 300/1000, Loss: 2.074924, Acc: 0.190430
Epoch 320/1000, Loss: 2.078058, Acc: 0.201172
Epoch 340/1000, Loss: 2.040591, Acc: 0.212891
Epoch 360/1000, Loss: 2.028210, Acc: 0.1

KeyboardInterrupt: 

In [6]:
for name, params in softmax_model.named_parameters():
    print(name, params.shape)


layers.0.attention.query_projection.weight torch.Size([32, 32])
layers.0.attention.query_projection.bias torch.Size([32])
layers.0.attention.key_projection.weight torch.Size([32, 32])
layers.0.attention.key_projection.bias torch.Size([32])
layers.0.attention.value_projection.weight torch.Size([32, 32])
layers.0.attention.value_projection.bias torch.Size([32])
layers.0.attention.out_projection.weight torch.Size([32, 32])
layers.0.attention.out_projection.bias torch.Size([32])
layers.0.linear1.weight torch.Size([128, 32])
layers.0.linear1.bias torch.Size([128])
layers.0.linear2.weight torch.Size([32, 128])
layers.0.linear2.bias torch.Size([32])
layers.0.norm1.weight torch.Size([32])
layers.0.norm1.bias torch.Size([32])
layers.0.norm2.weight torch.Size([32])
layers.0.norm2.bias torch.Size([32])
layers.1.attention.query_projection.weight torch.Size([32, 32])
layers.1.attention.query_projection.bias torch.Size([32])
layers.1.attention.key_projection.weight torch.Size([32, 32])
layers.1.atte

In [11]:
softmax_model = torch.nn.Sequential(softmax_model, torch.nn.Linear(embed_dim, 10//2)).to('cuda')

In [12]:
for name, params in softmax_model.named_parameters():
    print(name, params.shape)

0.layers.0.attention.query_projection.weight torch.Size([32, 32])
0.layers.0.attention.query_projection.bias torch.Size([32])
0.layers.0.attention.key_projection.weight torch.Size([32, 32])
0.layers.0.attention.key_projection.bias torch.Size([32])
0.layers.0.attention.value_projection.weight torch.Size([32, 32])
0.layers.0.attention.value_projection.bias torch.Size([32])
0.layers.0.attention.out_projection.weight torch.Size([32, 32])
0.layers.0.attention.out_projection.bias torch.Size([32])
0.layers.0.linear1.weight torch.Size([128, 32])
0.layers.0.linear1.bias torch.Size([128])
0.layers.0.linear2.weight torch.Size([32, 128])
0.layers.0.linear2.bias torch.Size([32])
0.layers.0.norm1.weight torch.Size([32])
0.layers.0.norm1.bias torch.Size([32])
0.layers.0.norm2.weight torch.Size([32])
0.layers.0.norm2.bias torch.Size([32])
0.layers.1.attention.query_projection.weight torch.Size([32, 32])
0.layers.1.attention.query_projection.bias torch.Size([32])
0.layers.1.attention.key_projection.wei

In [13]:
import torch, os, sys
import numpy as np
torch.backends.cuda.enable_mem_efficient_sdp(False)
torch.backends.cuda.enable_flash_sdp(False)
torch.backends.cuda.enable_math_sdp(True)
os.environ['CUDA_VISIBLE_DEVICES'] = '3'
import pandas as pd
from fast_transformers.builders import TransformerEncoderBuilder
import torch.nn.functional as F
import torch.nn as nn

# Parameters
n_layers = 2
num_heads = 1
embed_dim = 32
n_hid = 128
vocab_size = 10
device = 'cuda'

# Transformer builder
torch.manual_seed(0)
builder = TransformerEncoderBuilder.from_kwargs(
    n_layers=n_layers,
    n_heads=num_heads,
    query_dimensions=embed_dim // num_heads,
    value_dimensions=embed_dim // num_heads,
    feed_forward_dimensions=n_hid,
    dropout=0.1
)

# Positional Encoding Class
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, dropout=0.1, max_len=5000, device='cuda'):
        super(PositionalEncoding, self).__init__()
        self.dropout = nn.Dropout(p=dropout)
        pe = torch.zeros(max_len, d_model, device=device)
        position = torch.arange(0, max_len, device=device).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2, device=device) * -(np.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position.float() * div_term)
        pe[:, 1::2] = torch.cos(position.float() * div_term)
        pe = pe.unsqueeze(0)  # Shape [1, max_len, d_model]
        self.register_buffer('pe', pe)

    def forward(self, x):
        x = x + self.pe[:, :x.size(1)]
        return self.dropout(x)

# Model Class
class AssociativeRecallModel(nn.Module):
    def __init__(self, vocab_size, embed_dim, builder):
        super(AssociativeRecallModel, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.pos_encoder = PositionalEncoding(embed_dim, device=device)
        self.transformer_encoder = builder.get()
        self.fc_out = nn.Linear(embed_dim, vocab_size // 2)
    
    def forward(self, input_seq):
        embedded = self.embedding(input_seq)  # Shape [batch_size, seq_len, embed_dim]
        embedded = self.pos_encoder(embedded)
        output = self.transformer_encoder(embedded)  # Shape [batch_size, seq_len, embed_dim]
        output = self.fc_out(output[:, -1, :])  # Use the output corresponding to the query
        return output

# Data Generation Function
def generate_associative_recall_batch(seq_len=10, batch_size=32, vocab_size=10, device='cuda'):
    """Generate a batch of associative recall training data with discrete tokens."""
    batch_indices = torch.arange(batch_size, device=device)
    kv_pairs = torch.randint(0, vocab_size // 2, (batch_size, vocab_size), device=device)
    
    keys = torch.randint(0, vocab_size // 2, (batch_size, seq_len), device=device)
    values = torch.gather(kv_pairs, 1, keys)
    
    # Randomly select query indices
    query_positions = torch.randint(0, seq_len, (batch_size,), device=device)
    queries = keys[batch_indices, query_positions]  # Queries are keys
    
    # True targets are the values corresponding to the queries
    targets = values[batch_indices, query_positions]  # Shape [batch_size]
    
    # Prepare input sequence: interleave keys and values, then append query
    input_seq = torch.zeros(batch_size, seq_len * 2, dtype=torch.long, device=device)
    input_seq[:, 0::2] = keys  # Even indices for keys
    input_seq[:, 1::2] = values + vocab_size // 2  # Shift values to a different index range
    
    input_seq = torch.cat([input_seq, queries.unsqueeze(1)], dim=1)  # Shape [batch_size, seq_len * 2 + 1]
    return input_seq, targets

# Training Function
def train_model(model, seq_len, vocab_size=10, n_epochs=1000, batch_size=1024, device='cuda'):
    """Train model on associative recall classification task."""
    optimizer = torch.optim.Adam(model.parameters(), lr=5e-4, weight_decay=0.001)
    criterion = torch.nn.CrossEntropyLoss()
    losses = []
    
    for epoch in range(n_epochs):
        model.train()
        input_seq, targets = generate_associative_recall_batch(seq_len, batch_size, vocab_size, device)
        
        optimizer.zero_grad()
        predictions = model(input_seq)  # Shape [batch_size, vocab_size // 2]
        
        loss = criterion(predictions, targets)
        loss.backward()
        optimizer.step()
        
        losses.append(loss.item())
        
        if (epoch + 1) % 20 == 0:
            pred_labels = predictions.argmax(dim=-1)
            acc = (pred_labels == targets).float().mean()
            print(f"Epoch {epoch+1}/{n_epochs}, Loss: {loss.item():.6f}, Acc: {acc.item():.6f}")
    
    return np.mean(losses[-10:])  # Return average of last 10 losses

# Evaluation Function
def evaluate_model(model, seq_len, vocab_size=10, n_batches=10, batch_size=32, device='cuda'):
    """Evaluate model on associative recall classification task."""
    model.eval()
    total_acc = 0
    
    with torch.no_grad():
        for _ in range(n_batches):
            input_seq, targets = generate_associative_recall_batch(seq_len, batch_size, vocab_size, device)
            predictions = model(input_seq)
            pred_labels = predictions.argmax(dim=-1)
            acc = (pred_labels == targets).float().mean()
            total_acc += acc.item()
    
    return total_acc / n_batches

# Run experiments
seq_lengths = [20]
results = {'linear': [], 'softmax': []}

print("\nTraining and evaluating models...")
print("\nSequence Length | Linear Acc | Softmax Acc")
print("-" * 45)

for seq_len in seq_lengths:
    # Reset models
    builder.attention_type = "full"
    softmax_model = AssociativeRecallModel(vocab_size, embed_dim, builder).to(device)
    
    builder.attention_type = "linear"
    linear_model = AssociativeRecallModel(vocab_size, embed_dim, builder).to(device)
    
    # Train models
    print(f"\nTraining on sequence length {seq_len}:")
    print("Linear attention model:")
    train_model(linear_model, seq_len, vocab_size)
    print("\nSoftmax attention model:")
    train_model(softmax_model, seq_len, vocab_size)
    
    # Evaluate models
    linear_eval_acc = evaluate_model(linear_model, seq_len, vocab_size)
    softmax_eval_acc = evaluate_model(softmax_model, seq_len, vocab_size)
    
    results['linear'].append(linear_eval_acc)
    results['softmax'].append(softmax_eval_acc)
    
    print(f"\n{seq_len:14d} | {linear_eval_acc:.6f} | {softmax_eval_acc:.6f}")

# Create DataFrame with results
results_df = pd.DataFrame({
    'seq_length': seq_lengths,
    'linear_acc': results['linear'],
    'softmax_acc': results['softmax']
})
print(results_df)



Training and evaluating models...

Sequence Length | Linear Acc | Softmax Acc
---------------------------------------------

Training on sequence length 20:
Linear attention model:
Epoch 20/1000, Loss: 1.633376, Acc: 0.187500
Epoch 40/1000, Loss: 1.621448, Acc: 0.199219
Epoch 60/1000, Loss: 1.609809, Acc: 0.215820
Epoch 80/1000, Loss: 1.619707, Acc: 0.199219
Epoch 100/1000, Loss: 1.619116, Acc: 0.178711
Epoch 120/1000, Loss: 1.615094, Acc: 0.211914
Epoch 140/1000, Loss: 1.612426, Acc: 0.181641
Epoch 160/1000, Loss: 1.613730, Acc: 0.197266
Epoch 180/1000, Loss: 1.610983, Acc: 0.193359
Epoch 200/1000, Loss: 1.613538, Acc: 0.196289
Epoch 220/1000, Loss: 1.615583, Acc: 0.179688
Epoch 240/1000, Loss: 1.614377, Acc: 0.190430
Epoch 260/1000, Loss: 1.611668, Acc: 0.197266
Epoch 280/1000, Loss: 1.612118, Acc: 0.203125
Epoch 300/1000, Loss: 1.615479, Acc: 0.184570
Epoch 320/1000, Loss: 1.612712, Acc: 0.190430
Epoch 340/1000, Loss: 1.613738, Acc: 0.194336
Epoch 360/1000, Loss: 1.612365, Acc: 0.1